What you need to deliver:
1. Choose a text dataset and load it
2. Tokenize and build a single vocabulary (not two, since source and target are the same language)
3. Create input-target pairs by shifting sequences by one token
4. Split into train/validation/test sets
5. Reuse the LSTMLayer, StackLSTMLayers, Encoder, Decoder, and Seq2Seq classes (with minor adjustments — both encoder and decoder now share the same vocabulary size)
6. Train the model
7. In the final cell, feed in 10 test inputs and print the model's predicted outputs


In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchtext.vocab import build_vocab_from_iterator
import spacy
import numpy as np
import random
import math
import time

In [26]:
SEED = 1234
# Setting a seed ensures reproducibility of results in random processes.
# By setting the seed to a fixed value, random number generation becomes deterministic,
# meaning that the same sequence of random numbers will be generated each time the code is run.

# Set the seed for Python's built-in random module
random.seed(SEED)

# Set the seed for NumPy's random number generator
np.random.seed(SEED)

# Set the seed for PyTorch's random number generator on CPU
torch.manual_seed(SEED)

# Check if CUDA (GPU acceleration) is available on the system, or mps
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')


In [27]:
class LSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        '''
          - Args:
                - input_size: The number of expected features in the input.
                - hidden_size: The number of features in the hidden state.
          - Functionality:
                - Initialises the LSTM layer with the specified input size, and hidden size.
                - Initialises weight and bias parameters.
        '''
        super(LSTMLayer, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Define parameters for a single LSTM layer
        # We mark the weight matrices as a parameter of the model using nn.Parameter
        # This is done to indicate that this tensor should be considered a model parameter.
        # So that backward function in pytorch consider these matrices during optimisation,
        # and to calculate the gradients with respect to them.
        self.W_f = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_f = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_f = nn.Parameter(torch.Tensor(hidden_size))

        self.W_i = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_i = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_i = nn.Parameter(torch.Tensor(hidden_size))

        self.W_g = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_g = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_g = nn.Parameter(torch.Tensor(hidden_size))

        self.W_o = nn.Parameter(torch.Tensor(input_size, hidden_size))
        self.U_o = nn.Parameter(torch.Tensor(hidden_size, hidden_size))
        self.b_o = nn.Parameter(torch.Tensor(hidden_size))

    def forward(self, input, h_prev, c_prev):
        # Concatenate input and previous hidden state

        # Forget gate
        # Now that you're becoming comfortable with matrix multiplication,
        # we have commented on what's happening with the sizes for only one equation.
        # We want you to comment on the other equations as well.
        # input: (batch_size, input_size) @ W_f: (input_size, hidden_size) -> (batch_size, hidden_size)
        # h_prev: (batch_size, hidden_size) @ U_f: (hidden_size, hidden_size) -> (batch_size, hidden_size)
        
        # input: (batch_size, input_size) @ W_f: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_f (hidden_size, hidden_size) +
        # b_f(hidden_size, ) -> (batch_size, hidden_size)
        f = torch.sigmoid(input @ self.W_f + h_prev @ self.U_f + self.b_f)

        # Element-wise multiplication requires same shape
        # f: (batch_size, hidden_size) * c_prev: (batch_size, hidden_size) -> (batch_size, hidden_size)
        k = f * c_prev #mask


        # Input gate (Add gate)
        # input: (batch_size, input_size) @ W_i: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_i: (hidden_size, hidden_size) +
        # b_i: (hidden_size, ) -> (batch_size, hidden_size)
        i = torch.sigmoid(input @ self.W_i + h_prev @ self.U_i + self.b_i)

        # input: (batch_size, input_size) @ W_g:(input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_g: (hidden_size, hidden_size) +
        # b_g: (hidden_size, ) -> (batch_size, hidden_size)
        g = torch.tanh(input @ self.W_g + h_prev @ self.U_g + self.b_g) # Candidate cell state

        # g: (batch_size, hidden_size) * i: (batch_size, hidden_size) -> (batch_size, hidden_size)
        j =  g * i #mask


        # input: (batch_size, input_size) @ W_o: (input_size, hidden_size) +
        # h_prev: (batch_size, hidden_size) @ U_o: (hidden_size, hidden_size) +
        # b_o: (hidden_size) -> (batch_size, hidden_size)
        # Output gate
        o = torch.sigmoid(input @ self.W_o + h_prev @ self.U_o + self.b_o)


        # Update cell state
        # j: (batch_size, hidden_size) + k: (batch_size, hidden_size) -> (batch_size, hidden_size)
        c_next = j + k

        # Update hidden state
        # o: (batch_size, hidden_size) * tanh(c_next): (batch_size, hidden_size) -> (batch_size, hidden_size)
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

In [28]:
class StackLSTMLayers(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        '''
          - Args:
                - input_size: The number of expected features in the input.
                - hidden_size: The number of features in the hidden state.
                - num_layers: Number of Stacked recurrent layers.
          - Functionality:
                - Initialises the LSTM layer with the specified input size, hidden size, and number of layers.
                - Initialises weight and bias parameters for each layer.
        '''
        super(StackLSTMLayers, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Create stack of LSTM layers if num_layers > 1
        self.layers = nn.ModuleList([LSTMLayer(input_size if i == 0 else hidden_size, hidden_size) for i in range(num_layers)])

    def forward(self, input, hidden=None):
        if hidden is None:
            # initialise hidden and cell states if not provided
            hidden = self.init_hidden(input.size(1))

        # Unpack hidden states
        hiddens, cells = hidden

        outputs = []

        # Iterate through each time step
        for input_t in input:
            # Iterate through each layer
            for layer_idx, layer in enumerate(self.layers):
                # Pass input through the current layer
                hiddens[layer_idx], cells[layer_idx] = layer(input_t, hiddens[layer_idx], cells[layer_idx])

                # Update input for the next layer (if any)
                if layer_idx < self.num_layers - 1:
                    input_t = hiddens[layer_idx]

            # Append output of current time step
            outputs.append(hiddens[-1])

        # Stack outputs along the sequence dimension
        outputs = torch.stack(outputs, dim=0)


        # Return outputs, hidden states, and cell states
        return outputs, (hiddens, cells)

    def init_hidden(self, batch_size):
        # initialise hidden and cell states for each layer
        hiddens = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.num_layers)]
        cells = [torch.zeros(batch_size, self.hidden_size, device=device) for _ in range(self.num_layers)]
        return hiddens, cells

In [29]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()

        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        self.rnn = StackLSTMLayers(emb_dim, hid_dim, n_layers)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        #src = [src len, batch size]

        embedded = self.dropout(self.embedding(src)) ## UNDERSTAND THIS

        #embedded = [src len, batch size, emb dim]

        outputs, (hidden, cell) = self.rnn(embedded)

        #outputs = [src len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #outputs are always from the top hidden layer
        return hidden, cell

In [30]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):

        super().__init__()

        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(output_dim, emb_dim)

        self.rnn = StackLSTMLayers(emb_dim, hid_dim, n_layers)

        self.fc_out = nn.Linear(hid_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):

        #input = [batch size]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #n directions in the decoder will both always be 1, therefore:
        #hidden = [n layers, batch size, hid dim]
        #context = [n layers, batch size, hid dim]

        input = input.unsqueeze(0)

        #input = [1, batch size]

        embedded = self.dropout(self.embedding(input))

        #embedded = [1, batch size, emb dim]

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        #output = [seq len, batch size, hid dim * n directions]
        #hidden = [n layers * n directions, batch size, hid dim]
        #cell = [n layers * n directions, batch size, hid dim]

        #seq len and n directions will always be 1 in the decoder, therefore:
        #output = [1, batch size, hid dim]
        #hidden = [n layers, batch size, hid dim]
        #cell = [n layers, batch size, hid dim]

        prediction = self.fc_out(output.squeeze(0))

        #prediction = [batch size, output dim]

        return prediction, hidden, cell

In [31]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim, \
            "Hidden dimensions of encoder and decoder must be equal!"
        assert encoder.n_layers == decoder.n_layers, \
            "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio = 0.5):

        #src = [src len, batch size]
        #trg = [trg len, batch size]
        #teacher_forcing_ratio is probability to use teacher forcing
        #e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time

        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        #tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        #last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)

        #first input to the decoder is the <sos> tokens
        input = trg[0,:]

        for t in range(1, trg_len):

            #insert input token embedding, previous hidden and previous cell states
            #receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)

            #place predictions in a tensor holding predictions for each token
            outputs[t] = output

            #decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            #get the highest predicted token from our predictions
            top1 = output.argmax(1)

            #if teacher forcing, use actual next token as next input
            #if not, use predicted token
            input = trg[t] if teacher_force else top1

        return outputs

### Load and Clean the Text

In [32]:
# Read the file as a string
with open("great_gatsby.txt", "r", encoding="utf-8") as f:
    book_as_string = f.read()

# Strip Gutenberg header/footer
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK THE GREAT GATSBY ***"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK THE GREAT GATSBY ***"

start_idx = book_as_string.find(start_marker) + len(start_marker)
end_idx = book_as_string.find(end_marker)

clean_text = book_as_string[start_idx:end_idx].strip()
clean_text = clean_text.replace('\n', ' ').replace('  ', ' ')

# Split into sentences
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 1500000  # Increase limit to handle longer texts
doc = nlp(clean_text)
# filtering for shorter sentences
sentences = [sent.text.strip() for sent in doc.sents 
             if 5 <= len(sent.text.strip().split()) <= 25]

### Tokenize and Build ONE Vocabulary
Need to understand this part better

In [33]:
def tokenize(text):
    return [tok.text.lower() for tok in nlp.tokenizer(text)]

# Yield tokens from sentences
def yield_tokens(sentences):
    for sentence in sentences:
        yield tokenize(sentence)

# Define special symbols and indices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

# Build ONE vocabulary from all sentences
vocab = build_vocab_from_iterator(yield_tokens(sentences),
                                  min_freq=2,
                                  specials=special_symbols,
                                  special_first=True)

vocab.set_default_index(UNK_IDX)

### Create a Dataset Class with Shifted Pairs

In [34]:
class LanguageModelDataset(Dataset):
    """
    Dataset for language modeling with shifted pairs.
    
    For each sentence [w1, w2, ..., wN], returns:
        src (input):  [<bos>, w1, w2, ..., wN]  - what the model reads
        trg (target): [w1, w2, ..., wN, <eos>]  - what the model predicts
    
    The target is the input shifted by one token.
    """
    
    def __init__(self, sentences, vocab, tokenize_fn):
        self.sentences = sentences
        self.vocab = vocab
        self.tokenize_fn = tokenize_fn
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        tokens = self.tokenize_fn(sentence)
        token_indices = self.vocab(tokens)  # Convert tokens to indices
        
        # Create shifted pairs
        src = [BOS_IDX] + token_indices          # [<bos>, w1, w2, ..., wN]
        trg = token_indices + [EOS_IDX]          # [w1, w2, ..., wN, <eos>]
        
        return torch.tensor(src, dtype=torch.long), torch.tensor(trg, dtype=torch.long)

### Split into Train/Validation/Test 

In [35]:
random.shuffle(sentences)
n = len(sentences)
train_sents = sentences[:int(0.8*n)]
valid_sents = sentences[int(0.8*n):int(0.9*n)]
test_sents  = sentences[int(0.9*n):]

### LanguageModelDataset objects

In [36]:
train_dataset = LanguageModelDataset(train_sents, vocab, tokenize)
valid_dataset = LanguageModelDataset(valid_sents, vocab, tokenize)
test_dataset = LanguageModelDataset(test_sents, vocab, tokenize)

In [37]:
print(len(train_dataset))

1763


### Collate Function

In [38]:
def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX)
    return src_batch, trg_batch

### Model Initialization

In [39]:
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 3
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2
BATCH_SIZE = 64

VOCAB_SIZE = len(vocab)  # single vocabulary
enc = Encoder(VOCAB_SIZE, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(VOCAB_SIZE, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 13,322,949 trainable parameters


In [40]:
optimizer = optim.Adam(model.parameters())

In [41]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [42]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    for src, trg in dataloader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        #trg = [trg len, batch size]
        #output = [trg len, batch size, output dim]

        output_dim = output.shape[-1]

        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)

        #trg = [(trg len - 1) * batch size]
        #output = [(trg len - 1) * batch size, output dim]

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / max(1, num_batches)

In [43]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    num_batches = 0
    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE, collate_fn=collate_fn)

    with torch.no_grad():

        for src, trg in dataloader:
            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg, 0) #turn off teacher forcing

            #trg = [trg len, batch size]
            #output = [trg len, batch size, output dim]

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            #trg = [(trg len - 1) * batch size]
            #output = [(trg len - 1) * batch size, output dim]

            loss = criterion(output, trg)

            epoch_loss += loss.item()
            num_batches += 1

    return epoch_loss / max(1, num_batches)

In [44]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [45]:
N_EPOCHS = 30
CLIP = 1

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_dataset, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_dataset, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'lstm-seq2seq-model.pt')

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Epoch: 01 | Time: 0m 12s
	Train Loss: 5.717 | Train PPL: 303.871
	 Val. Loss: 5.329 |  Val. PPL: 206.212
Epoch: 02 | Time: 0m 11s
	Train Loss: 5.312 | Train PPL: 202.773
	 Val. Loss: 5.304 |  Val. PPL: 201.083
Epoch: 03 | Time: 0m 11s
	Train Loss: 5.274 | Train PPL: 195.203
	 Val. Loss: 5.303 |  Val. PPL: 200.989
Epoch: 04 | Time: 0m 11s
	Train Loss: 5.250 | Train PPL: 190.609
	 Val. Loss: 5.301 |  Val. PPL: 200.559
Epoch: 05 | Time: 0m 11s
	Train Loss: 5.209 | Train PPL: 182.972
	 Val. Loss: 5.311 |  Val. PPL: 202.617
Epoch: 06 | Time: 0m 11s
	Train Loss: 5.129 | Train PPL: 168.910
	 Val. Loss: 5.288 |  Val. PPL: 197.971
Epoch: 07 | Time: 0m 11s
	Train Loss: 5.062 | Train PPL: 157.888
	 Val. Loss: 5.283 |  Val. PPL: 197.038
Epoch: 08 | Time: 0m 11s
	Train Loss: 5.003 | Train PPL: 148.856
	 Val. Loss: 5.270 |  Val. PPL: 194.512
Epoch: 09 | Time: 0m 11s
	Train Loss: 4.937 | Train PPL: 139.411
	 Val. Loss: 5.275 |  Val. PPL: 195.420
Epoch: 10 | Time: 0m 11s
	Train Loss: 4.896 | Train PPL

In [46]:
model.load_state_dict(torch.load('lstm-seq2seq-model.pt'))

loss = evaluate(model, valid_dataset, criterion)

print(f'| Loss: {loss:.3f} | PPL: {math.exp(loss):7.3f} |')
# min_freq = 2 -> | Loss: 4.852 | PPL: 128.047 |

| Loss: 4.859 | PPL: 128.887 |


### Step 10: Inference — 10 Test Inputs

This is the final deliverable cell.
1. Pick 10 sentences from test set
2. For each, tokenize and convert to indices to create the src tensor
3. Run the encoder to get the context vector
4. Run the decoder token-by-token (with teacher_forcing_ratio=0) to generate predictions
5. Convert the predicted indices back to words using the vocabulary
6. Print the input sentence and the model's predicted continuation

## Step 11: Improved Training + Inference Pipeline (Plan Implementation)

This section implements:
- train-only vocabulary (no validation/test leakage)
- explicit decoder input/target alignment
- true autoregressive generation from `<bos>`
- greedy and sampling decode strategies
- model-size ablation and final selection

In [ ]:
# ---------- Rebuild split + vocab (train only) ----------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

all_sentences = sentences[:]  # keep same preprocessing/tokenization from earlier cells
random.shuffle(all_sentences)

n = len(all_sentences)
train_sents = all_sentences[:int(0.8 * n)]
valid_sents = all_sentences[int(0.8 * n):int(0.9 * n)]
test_sents = all_sentences[int(0.9 * n):]

UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]

def yield_tokens(text_list):
    for s in text_list:
        yield tokenize(s)

vocab = build_vocab_from_iterator(
    yield_tokens(train_sents),
    min_freq=1,
    specials=special_symbols,
    special_first=True,
)
vocab.set_default_index(UNK_IDX)
itos = vocab.get_itos()

# ---------- OOV diagnostics ----------
def compute_oov_rate(text_list, vocab_obj, tokenize_fn):
    total = 0
    unk = 0
    for s in text_list:
        toks = tokenize_fn(s)
        ids = vocab_obj(toks)
        total += len(ids)
        unk += sum(1 for x in ids if x == UNK_IDX)
    return 0.0 if total == 0 else unk / total

train_oov = compute_oov_rate(train_sents, vocab, tokenize)
valid_oov = compute_oov_rate(valid_sents, vocab, tokenize)
test_oov = compute_oov_rate(test_sents, vocab, tokenize)

print(f"Split sizes -> train: {len(train_sents)}, valid: {len(valid_sents)}, test: {len(test_sents)}")
print(f"Vocab size (train-only, min_freq=1): {len(vocab)}")
print(f"OOV rates -> train: {train_oov:.4f}, valid: {valid_oov:.4f}, test: {test_oov:.4f}")

# ---------- Dataset with explicit decoder input and target output ----------
class AlignedLanguageModelDataset(Dataset):
    """
    Returns:
      src:        [<bos>, w1, ..., wN]
      decoder_in: [<bos>, w1, ..., wN]
      target_out: [w1, ..., wN, <eos>]
    """

    def __init__(self, text_list, vocab_obj, tokenize_fn):
        self.text_list = text_list
        self.vocab = vocab_obj
        self.tokenize_fn = tokenize_fn

    def __len__(self):
        return len(self.text_list)

    def __getitem__(self, idx):
        toks = self.tokenize_fn(self.text_list[idx])
        ids = self.vocab(toks)

        src = [BOS_IDX] + ids
        decoder_in = [BOS_IDX] + ids
        target_out = ids + [EOS_IDX]

        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(decoder_in, dtype=torch.long),
            torch.tensor(target_out, dtype=torch.long),
        )


def collate_aligned(batch):
    src_batch, dec_in_batch, tgt_out_batch = [], [], []
    for src, dec_in, tgt_out in batch:
        src_batch.append(src)
        dec_in_batch.append(dec_in)
        tgt_out_batch.append(tgt_out)

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    dec_in_batch = pad_sequence(dec_in_batch, padding_value=PAD_IDX)
    tgt_out_batch = pad_sequence(tgt_out_batch, padding_value=PAD_IDX)
    return src_batch, dec_in_batch, tgt_out_batch


train_dataset_v2 = AlignedLanguageModelDataset(train_sents, vocab, tokenize)
valid_dataset_v2 = AlignedLanguageModelDataset(valid_sents, vocab, tokenize)
test_dataset_v2 = AlignedLanguageModelDataset(test_sents, vocab, tokenize)

# ---------- Seq2Seq with corrected step alignment ----------
class Seq2SeqAligned(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

        assert encoder.hid_dim == decoder.hid_dim
        assert encoder.n_layers == decoder.n_layers

    def forward(self, src, decoder_in, teacher_forcing_ratio=0.5):
        # src: [src_len, batch]
        # decoder_in: [dec_len, batch] where decoder_in[0] should be <bos>
        batch_size = decoder_in.shape[1]
        dec_len = decoder_in.shape[0]
        vocab_size = self.decoder.output_dim

        outputs = torch.zeros(dec_len, batch_size, vocab_size, device=self.device)

        hidden, cell = self.encoder(src)
        input_tok = decoder_in[0, :]  # <bos>

        for t in range(dec_len):
            output, hidden, cell = self.decoder(input_tok, hidden, cell)
            outputs[t] = output

            if t < dec_len - 1:
                teacher_force = random.random() < teacher_forcing_ratio
                top1 = output.argmax(1)
                input_tok = decoder_in[t + 1] if teacher_force else top1

        return outputs


# ---------- Training / evaluation ----------
def train_epoch_aligned(model, dataset, optimizer, criterion, clip, batch_size):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_aligned, shuffle=True)

    for src, decoder_in, target_out in dataloader:
        src = src.to(device)
        decoder_in = decoder_in.to(device)
        target_out = target_out.to(device)

        optimizer.zero_grad()
        output = model(src, decoder_in, teacher_forcing_ratio=0.5)

        output_dim = output.shape[-1]
        loss = criterion(output.view(-1, output_dim), target_out.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    return epoch_loss / max(1, num_batches)


def evaluate_aligned(model, dataset, criterion, batch_size):
    model.eval()
    epoch_loss = 0.0
    num_batches = 0

    dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_aligned)

    with torch.no_grad():
        for src, decoder_in, target_out in dataloader:
            src = src.to(device)
            decoder_in = decoder_in.to(device)
            target_out = target_out.to(device)

            output = model(src, decoder_in, teacher_forcing_ratio=0.0)
            output_dim = output.shape[-1]

            loss = criterion(output.view(-1, output_dim), target_out.view(-1))
            epoch_loss += loss.item()
            num_batches += 1

    return epoch_loss / max(1, num_batches)


# ---------- Decoding helpers ----------
def apply_top_k(logits, k):
    if k is None or k <= 0 or k >= logits.size(-1):
        return logits
    top_vals, _ = torch.topk(logits, k)
    kth = top_vals[:, -1].unsqueeze(1)
    return torch.where(logits < kth, torch.full_like(logits, -1e10), logits)


def apply_top_p(logits, top_p):
    if top_p is None or top_p <= 0.0 or top_p >= 1.0:
        return logits

    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    probs = torch.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(probs, dim=-1)

    remove_mask = cum_probs > top_p
    remove_mask[:, 1:] = remove_mask[:, :-1].clone()
    remove_mask[:, 0] = False

    sorted_logits = sorted_logits.masked_fill(remove_mask, -1e10)

    filtered = torch.full_like(logits, -1e10)
    filtered.scatter_(1, sorted_idx, sorted_logits)
    return filtered


def apply_repetition_penalty(logits, generated_ids, penalty=1.1):
    if penalty is None or penalty <= 1.0 or len(generated_ids) == 0:
        return logits

    adjusted = logits.clone()
    used = torch.tensor(list(set(generated_ids)), device=logits.device)
    # reduce score for previously used tokens to limit loops
    adjusted[:, used] = adjusted[:, used] / penalty
    return adjusted


@torch.no_grad()
def generate(
    model,
    src_tensor,
    bos_idx,
    eos_idx,
    max_len=40,
    strategy="greedy",
    temperature=1.0,
    top_k=None,
    top_p=None,
    repetition_penalty=1.0,
):
    model.eval()

    hidden, cell = model.encoder(src_tensor)

    batch_size = src_tensor.shape[1]
    input_tok = torch.full((batch_size,), bos_idx, dtype=torch.long, device=src_tensor.device)

    generated = [[] for _ in range(batch_size)]
    finished = [False] * batch_size

    for _ in range(max_len):
        logits, hidden, cell = model.decoder(input_tok, hidden, cell)

        if strategy == "greedy":
            next_tok = logits.argmax(dim=1)
        else:
            step_logits = logits
            if temperature is not None and temperature > 0:
                step_logits = step_logits / temperature

            # apply per-batch repetition penalty
            for b in range(batch_size):
                step_logits[b:b+1] = apply_repetition_penalty(
                    step_logits[b:b+1],
                    generated[b],
                    penalty=repetition_penalty,
                )

            step_logits = apply_top_k(step_logits, top_k)
            step_logits = apply_top_p(step_logits, top_p)

            probs = torch.softmax(step_logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(1)

        input_tok = next_tok

        for b in range(batch_size):
            if not finished[b]:
                tid = int(next_tok[b].item())
                generated[b].append(tid)
                if tid == eos_idx:
                    finished[b] = True

        if all(finished):
            break

    return generated


# ---------- Training runner + ablation ----------
def build_model(config):
    emb_dim = config["emb_dim"]
    hid_dim = config["hid_dim"]
    n_layers = config["n_layers"]
    enc_dropout = config.get("enc_dropout", 0.2)
    dec_dropout = config.get("dec_dropout", 0.2)

    enc = Encoder(len(vocab), emb_dim, hid_dim, n_layers, enc_dropout)
    dec = Decoder(len(vocab), emb_dim, hid_dim, n_layers, dec_dropout)
    model = Seq2SeqAligned(enc, dec, device).to(device)

    def init_weights(m):
        for _, p in m.named_parameters():
            if p.requires_grad:
                nn.init.uniform_(p.data, -0.08, 0.08)

    model.apply(init_weights)
    return model


def run_experiment(config, n_epochs=30, clip=1.0):
    model = build_model(config)
    optimizer = optim.Adam(model.parameters(), lr=config.get("lr", 1e-3))
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    best_valid_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(n_epochs):
        train_loss = train_epoch_aligned(
            model,
            train_dataset_v2,
            optimizer,
            criterion,
            clip=clip,
            batch_size=config.get("batch_size", 64),
        )
        valid_loss = evaluate_aligned(
            model,
            valid_dataset_v2,
            criterion,
            batch_size=config.get("batch_size", 64),
        )

        history.append((train_loss, valid_loss))

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"[{config['name']}] Epoch {epoch + 1:02d} | "
                f"Train PPL: {math.exp(train_loss):.2f} | Valid PPL: {math.exp(valid_loss):.2f}"
            )

    model.load_state_dict(best_state)

    return {
        "name": config["name"],
        "config": config,
        "model": model,
        "best_valid_loss": best_valid_loss,
        "best_valid_ppl": math.exp(best_valid_loss),
        "history": history,
    }


def tokens_from_ids(ids, itos_list):
    out = []
    for idx in ids:
        tok = itos_list[idx]
        if tok == "<eos>":
            break
        out.append(tok)
    return out


def generated_unk_rate(results_tokens):
    total = sum(len(x) for x in results_tokens)
    if total == 0:
        return 0.0
    unk = sum(tok == "<unk>" for seq in results_tokens for tok in seq)
    return unk / total

print("Improved pipeline definitions loaded.")

In [ ]:
# ---------- Baseline snapshot (from previous section outputs) ----------
# Existing notebook baseline (before this refactor):
# valid loss ~ 4.859, valid PPL ~ 128.887
legacy_baseline = {
    "name": "legacy_notebook_baseline",
    "valid_loss": 4.859,
    "valid_ppl": math.exp(4.859),
}
print("Legacy baseline loaded:", legacy_baseline)

# ---------- Controlled ablation ----------
experiment_configs = [
    {
        "name": "baseline_256_512_l3",
        "emb_dim": 256,
        "hid_dim": 512,
        "n_layers": 3,
        "batch_size": 64,
        "lr": 1e-3,
        "enc_dropout": 0.2,
        "dec_dropout": 0.2,
    },
    {
        "name": "small_128_256_l2",
        "emb_dim": 128,
        "hid_dim": 256,
        "n_layers": 2,
        "batch_size": 64,
        "lr": 1e-3,
        "enc_dropout": 0.2,
        "dec_dropout": 0.2,
    },
    {
        "name": "small_128_256_l1",
        "emb_dim": 128,
        "hid_dim": 256,
        "n_layers": 1,
        "batch_size": 64,
        "lr": 1e-3,
        "enc_dropout": 0.2,
        "dec_dropout": 0.2,
    },
]

# Keep same training protocol across configs.
N_EPOCHS_ABLATION = 30

results = []
for cfg in experiment_configs:
    print("\n" + "=" * 80)
    print("Running:", cfg["name"])
    print("=" * 80)
    exp_result = run_experiment(cfg, n_epochs=N_EPOCHS_ABLATION, clip=1.0)
    print(
        f"Best valid loss: {exp_result['best_valid_loss']:.4f} | "
        f"Best valid PPL: {exp_result['best_valid_ppl']:.2f}"
    )
    results.append(exp_result)

results_sorted = sorted(results, key=lambda x: x["best_valid_loss"])
best_result = results_sorted[0]
best_model = best_result["model"]

print("\n" + "#" * 80)
print("Ablation summary")
print("#" * 80)
print(f"Legacy baseline valid PPL: {legacy_baseline['valid_ppl']:.2f}")
for r in results_sorted:
    print(
        f"{r['name']}: valid_loss={r['best_valid_loss']:.4f}, "
        f"valid_ppl={r['best_valid_ppl']:.2f}"
    )
print(f"Selected best model: {best_result['name']}")

In [ ]:
# ---------- Qualitative generation on fixed 10 prompts ----------
# use 10 deterministic prompts for all decoding modes
fixed_test_prompts = [s for s in test_sents if len(tokenize(s)) >= 5][:10]


def build_src_tensor(sentence, vocab_obj):
    token_ids = vocab_obj(tokenize(sentence))
    return torch.tensor([BOS_IDX] + token_ids, dtype=torch.long).unsqueeze(1).to(device)


def run_generation_set(model, prompts, strategy, temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0, max_len=40):
    generated_texts = []
    for sent in prompts:
        src_tensor = build_src_tensor(sent, vocab)
        gen_ids = generate(
            model,
            src_tensor,
            bos_idx=BOS_IDX,
            eos_idx=EOS_IDX,
            max_len=max_len,
            strategy=strategy,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
        )[0]
        gen_tokens = tokens_from_ids(gen_ids, itos)
        generated_texts.append(gen_tokens)
    return generated_texts


modes = [
    {"name": "greedy", "strategy": "greedy", "temperature": 1.0, "top_k": None, "top_p": None, "repetition_penalty": 1.0},
    {"name": "sample_topk20_t0.9", "strategy": "sample", "temperature": 0.9, "top_k": 20, "top_p": None, "repetition_penalty": 1.1},
    {"name": "sample_topp0.9_t0.9", "strategy": "sample", "temperature": 0.9, "top_k": None, "top_p": 0.9, "repetition_penalty": 1.1},
]

mode_outputs = {}
for mode in modes:
    toks = run_generation_set(
        best_model,
        fixed_test_prompts,
        strategy=mode["strategy"],
        temperature=mode["temperature"],
        top_k=mode["top_k"],
        top_p=mode["top_p"],
        repetition_penalty=mode["repetition_penalty"],
        max_len=40,
    )
    mode_outputs[mode["name"]] = toks

print("=" * 90)
print(f"BEST MODEL: {best_result['name']} | Valid PPL: {best_result['best_valid_ppl']:.2f}")
print("=" * 90)

for mode in modes:
    name = mode["name"]
    unk_rate = generated_unk_rate(mode_outputs[name])
    print(f"\n--- Decode mode: {name} | Generated <unk> rate: {unk_rate:.4f} ---")
    for i, sent in enumerate(fixed_test_prompts):
        print(f"\nTest {i+1}")
        print("Input:    ", sent)
        print("Target:   ", " ".join(tokenize(sent) + ["<eos>"]))
        print("Predicted:", " ".join(mode_outputs[name][i] + ["<eos>"]))

# Final acceptance signal
best_vs_legacy = best_result["best_valid_ppl"] < legacy_baseline["valid_ppl"]
print("\n" + "#" * 90)
print("Acceptance checks")
print("#" * 90)
print(f"Improved valid PPL vs legacy baseline: {best_vs_legacy}")
print("Review sampled outputs above for coherence/lexical variety and reduced <unk> loops.")

In [47]:
# Build reverse vocabulary mapping (index -> token)
itos = vocab.get_itos()

model.eval()

# Pick 10 test sentences with at least 5 tokens (to get meaningful predictions)
test_samples = [s for s in test_sents if len(tokenize(s)) >= 5][:10]

print("=" * 80)
print("INFERENCE: 10 Test Inputs from Moby Dick")
print("=" * 80)

with torch.no_grad():
    for i, sentence in enumerate(test_samples):
        # Tokenize and convert to vocabulary indices
        tokens = tokenize(sentence)
        token_indices = vocab(tokens)

        # Create src and trg tensors matching the dataset format
        # src: [<bos>, w1, w2, ..., wN]   shape: [src_len, 1]
        # trg: [w1, w2, ..., wN, <eos>]   shape: [trg_len, 1]
        src_tensor = torch.tensor(
            [BOS_IDX] + token_indices, dtype=torch.long
        ).unsqueeze(1).to(device)

        trg_tensor = torch.tensor(
            token_indices + [EOS_IDX], dtype=torch.long
        ).unsqueeze(1).to(device)

        # Forward pass with teacher forcing OFF (model uses only its own predictions)
        output = model(src_tensor, trg_tensor, teacher_forcing_ratio=0)

        # output shape: [trg_len, 1, vocab_size]
        # Position 0 is zeros (decoder loop starts at t=1)
        # Positions 1..N contain predictions for w2, w3, ..., wN, <eos>
        predicted_indices = output[1:].argmax(2).squeeze(1).cpu().tolist()
        predicted_tokens = [itos[idx] for idx in predicted_indices]

        # Ground truth target for comparison
        target_tokens = tokens[1:] + ["<eos>"]

        print(f"\nTest {i+1}:")
        print(f"  Input:     {sentence}")
        print(f"  Target:    {' '.join(target_tokens)}")
        print(f"  Predicted: {' '.join(predicted_tokens)}")

INFERENCE: 10 Test Inputs from Moby Dick

Test 1:
  Input:     and I said: ‘All right, Katspaugh, don’t pay him a penny till he shuts his mouth.’
  Target:    i said : ‘ all right , katspaugh , do n’t pay him a penny till he shuts his mouth . ’ <eos>
  Predicted: i had to <unk> — ” “ <unk> , “ i i had n’t to a <unk> of the . . <eos> <eos>

Test 2:
  Input:     He was reluctant to close the book, reading each item aloud and then looking eagerly at me.
  Target:    was reluctant to close the book , reading each item aloud and then looking eagerly at me . <eos>
  Predicted: was a <unk> , and <unk> <unk> <unk> , and <unk> <unk> <unk> <unk> <unk> <unk> . . <eos>

Test 3:
  Input:     “I’d like to be out there with him for about an hour.”
  Target:    i ’d like to be out there with him for about an hour . ” <eos>
  Predicted: i ’m n’t a <unk> — ” he said , the <unk> . . <eos> <eos>

Test 4:
  Input:     “I see you’re looking at my cuff buttons.”
  Target:    i see you ’re looking at my cuff 